In [1]:
import pandas as pd
import os
import glob
import json
from google.cloud import bigquery
from io import BytesIO
from datetime import datetime

In [2]:
# --- 1. CONFIGURAÇÕES DE ACESSO ---
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "theme-park-queue-data-f2e1d4785d38.json"
TABLE_ID = "theme-park-queue-data.theme_park_queues.historical-data"
BASE_PATH = r"C:\Users\bairru\Desktop\BCW\bcw-queue\data"

# Intervalo de segurança (ajuste se necessário)
START_LIMIT = datetime(2026, 1, 1)
END_LIMIT = datetime(2026, 5, 11)

In [3]:
# Dicionário para mapear o nome do arquivo ao ID e Nome correto do Parque
PARK_MAP = {
    "thorpe_park": {"id": 2, "name": "Thorpe Park", "tz": "Europe/London"},
    "disneyland_paris": {"id": 4, "name": "Disneyland Paris", "tz": "Europe/Paris"},
    "epcot": {"id": 5, "name": "Epcot", "tz": "America/New_York"},
    "magic_kingdom": {"id": 6, "name": "Magic Kingdom", "tz": "America/New_York"},
    "hollywood_studios": {"id": 7, "name": "Hollywood Studios", "tz": "America/New_York"},
    "animal_kingdom": {"id": 8, "name": "Animal Kingdom", "tz": "America/New_York"},
    "parc_asterix": {"id": 9, "name": "Parc Asterix", "tz": "Europe/Paris"},
    "hersheypark": {"id": 15, "name": "Hersheypark", "tz": "America/New_York"},
    "disneyland": {"id": 16, "name": "Disneyland", "tz": "America/Los_Angeles"},
    "california_adventure": {"id": 17, "name": "California Adventure", "tz": "America/Los_Angeles"},
    "seaworld_orlando": {"id": 21, "name": "Seaworld Orlando", "tz": "America/New_York"},
    "busch_gardens_tampa": {"id": 24, "name": "Busch Gardens Tampa", "tz": "America/New_York"},
    "walt_disney_studios_paris": {"id": 28, "name": "Walt Disney Studios Paris", "tz": "Europe/Paris"},
    "six_flags_magic_mountain": {"id": 32, "name": "Six Flags Magic Mountain", "tz": "America/Los_Angeles"},
    "knott's_berry_farm": {"id": 61, "name": "Knott's Berry Farm", "tz": "America/Los_Angeles"},
    "islands_of_adventure": {"id": 64, "name": "Islands Of Adventure", "tz": "America/New_York"},
    "universal_studios_orlando": {"id": 65, "name": "Universal Studios Orlando", "tz": "America/New_York"},
    "universal_studios_hollywood": {"id": 66, "name": "Universal Studios Hollywood", "tz": "America/Los_Angeles"},
    "epic_universe": {"id": 334, "name": "Epic Universe", "tz": "America/New_York"}
}

In [4]:
def upload_to_bq(df):
    if df.empty: return
    client = bigquery.Client()
    
    # Formato JSONL para evitar erros de tipos de dados e limites de streaming
    jsonl_data = df.to_json(orient='records', lines=True)
    file_obj = BytesIO(jsonl_data.encode('utf-8'))
    
    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
        write_disposition="WRITE_APPEND",
    )
    
    load_job = client.load_table_from_file(file_obj, TABLE_ID, job_config=job_config)
    load_job.result() # Aguarda o fim do processamento

In [5]:
if __name__ == "__main__":
    # Busca recursiva em todas as subpastas de data
    path_pattern = os.path.join(BASE_PATH, "**", "*.csv")
    all_files = glob.glob(path_pattern, recursive=True)
    
    print(f"🔍 Iniciando processamento de {len(all_files)} arquivos...")

    for file_path in all_files:
        filename = os.path.basename(file_path).lower()
        
        # 1. Filtro: Ignorar Beto Carrero (já feito)
        if "beto" in filename:
            continue
            
        # 2. Identificar Parque e Data
        config = None
        for key, val in PARK_MAP.items():
            if key in filename:
                config = val
                break
        
        if not config:
            continue

        try:
            # Tentar extrair a data do nome do arquivo (formato: parque_AAAA-MM-DD.csv)
            date_part = filename.split('_')[-1].replace('.csv', '')
            file_date = datetime.strptime(date_part, "%Y-%m-%d")
            
            # Filtro de intervalo solicitado
            if not (START_LIMIT <= file_date <= END_LIMIT):
                continue

            # 3. Processamento dos dados
            df = pd.read_csv(file_path)
            
            # Mapear colunas do CSV antigo para o esquema do BigQuery
            df = df.rename(columns={
                'id': 'ride_id',
                'name': 'ride_name',
                'collected_at': 'timestamp_utc'
            })
            
            # Adicionar metadados que não existiam nos CSVs individuais
            df['park_id'] = config['id']
            df['park_name'] = config['name']
            df['timezone'] = config['tz']
            
            # Preencher coluna 'land' se não existir
            if 'land' not in df.columns:
                df['land'] = 'N/A'
            else:
                df['land'] = df['land'].fillna('N/A')

            # Ordenar e selecionar apenas as colunas oficiais da tabela
            df_final = df[[
                "park_id", "park_name", "land", "ride_id", 
                "ride_name", "wait_time", "is_open", 
                "timestamp_utc", "timezone"
            ]]

            # Enviar para o BigQuery
            upload_to_bq(df_final)
            print(f"✅ Sucesso: {filename}")
            
        except Exception as e:
            print(f"❌ Erro ao processar {filename}: {e}")

    print("\n🏁 Processo concluído! Verifique seu Dashboard.")

🔍 Iniciando processamento de 3887 arquivos...
✅ Sucesso: animal_kingdom_2026-01-01.csv
✅ Sucesso: animal_kingdom_2026-01-02.csv
✅ Sucesso: animal_kingdom_2026-01-03.csv
✅ Sucesso: animal_kingdom_2026-01-04.csv
✅ Sucesso: animal_kingdom_2026-01-05.csv
✅ Sucesso: animal_kingdom_2026-01-06.csv
✅ Sucesso: animal_kingdom_2026-01-07.csv
✅ Sucesso: animal_kingdom_2026-01-08.csv
✅ Sucesso: animal_kingdom_2026-01-09.csv
✅ Sucesso: animal_kingdom_2026-01-10.csv
✅ Sucesso: animal_kingdom_2026-01-11.csv
✅ Sucesso: animal_kingdom_2026-01-12.csv
✅ Sucesso: animal_kingdom_2026-01-13.csv
✅ Sucesso: animal_kingdom_2026-01-14.csv
✅ Sucesso: animal_kingdom_2026-01-15.csv
✅ Sucesso: animal_kingdom_2026-01-16.csv
✅ Sucesso: animal_kingdom_2026-01-17.csv
✅ Sucesso: animal_kingdom_2026-01-18.csv
✅ Sucesso: animal_kingdom_2026-01-19.csv
✅ Sucesso: animal_kingdom_2026-01-20.csv
✅ Sucesso: animal_kingdom_2026-01-21.csv
✅ Sucesso: animal_kingdom_2026-01-22.csv
✅ Sucesso: animal_kingdom_2026-01-23.csv
✅ Sucesso: 

KeyboardInterrupt: 

In [11]:
import pandas as pd
import os
import glob
import json
from google.cloud import bigquery
from io import BytesIO
from datetime import datetime

# --- 1. CONFIGURAÇÕES DE ACESSO ---
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "theme-park-queue-data-f2e1d4785d38.json"

# IMPORTANTE: Usando a tabela temporária para burlar o erro 403
TABLE_ID = "theme-park-queue-data.theme_park_queues.historical-data"
BASE_PATH = r"C:\Users\bairru\Desktop\BCW\bcw-queue\data"

PARK_MAP = {
    "thorpe_park": {"id": 2, "name": "Thorpe Park", "tz": "Europe/London"},
    "disneyland_paris": {"id": 4, "name": "Disneyland Paris", "tz": "Europe/Paris"},
    "epcot": {"id": 5, "name": "Epcot", "tz": "America/New_York"},
    "magic_kingdom": {"id": 6, "name": "Magic Kingdom", "tz": "America/New_York"},
    "hollywood_studios": {"id": 7, "name": "Hollywood Studios", "tz": "America/New_York"},
    "animal_kingdom": {"id": 8, "name": "Animal Kingdom", "tz": "America/New_York"},
    "parc_asterix": {"id": 9, "name": "Parc Asterix", "tz": "Europe/Paris"},
    "hersheypark": {"id": 15, "name": "Hersheypark", "tz": "America/New_York"},
    "disneyland": {"id": 16, "name": "Disneyland", "tz": "America/Los_Angeles"},
    "california_adventure": {"id": 17, "name": "California Adventure", "tz": "America/Los_Angeles"},
    "seaworld_orlando": {"id": 21, "name": "Seaworld Orlando", "tz": "America/New_York"},
    "busch_gardens_tampa": {"id": 24, "name": "Busch Gardens Tampa", "tz": "America/New_York"},
    "walt_disney_studios_paris": {"id": 28, "name": "Walt Disney Studios Paris", "tz": "Europe/Paris"},
    "six_flags_magic_mountain": {"id": 32, "name": "Six Flags Magic Mountain", "tz": "America/Los_Angeles"},
    "knott's_berry_farm": {"id": 61, "name": "Knott's Berry Farm", "tz": "America/Los_Angeles"},
    "islands_of_adventure": {"id": 64, "name": "Islands Of Adventure", "tz": "America/New_York"},
    "universal_studios_orlando": {"id": 65, "name": "Universal Studios Orlando", "tz": "America/New_York"},
    "universal_studios_hollywood": {"id": 66, "name": "Universal Studios Hollywood", "tz": "America/Los_Angeles"},
    "epic_universe": {"id": 334, "name": "Epic Universe", "tz": "America/New_York"}
}

def upload_batch_to_bq(df_list):
    if not df_list: return
    
    client = bigquery.Client()
    df_combined = pd.concat(df_list, ignore_index=True)
    
    jsonl_data = df_combined.to_json(orient='records', lines=True)
    file_obj = BytesIO(jsonl_data.encode('utf-8'))
    
    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
        write_disposition="WRITE_APPEND",
        autodetect=True,
    )
    
    try:
        load_job = client.load_table_from_file(file_obj, TABLE_ID, job_config=job_config)
        load_job.result()
        print(f"📦 Lote de {len(df_list)} arquivos carregado ({len(df_combined)} linhas).")
    except Exception as e:
        print(f"❌ Erro crítico no upload do lote: {e}")

if __name__ == "__main__":
    path_pattern = os.path.join(BASE_PATH, "**", "*.csv")
    all_files = glob.glob(path_pattern, recursive=True)
    
    print(f"🔍 Analisando {len(all_files)} arquivos...")

    batch_list = []
    processed_count = 0

    for file_path in all_files:
        filename = os.path.basename(file_path).lower()
        
        if "beto" in filename:
            continue
            
        config = None
        park_key = None
        for key, val in PARK_MAP.items():
            if key in filename:
                config = val
                park_key = key
                break
        
        if not config:
            continue

        try:
            date_part = filename.split('_')[-1].replace('.csv', '')
            file_date = datetime.strptime(date_part, "%Y-%m-%d")
            
            # --- LÓGICA DE FILTRO ESPECÍFICA QUE VOCÊ IDENTIFICOU ---
            should_load = False
            
            # 1. Hollywood Studios: 24/03 a 11/05
            if park_key == "universal_studios_hollywood":
                if datetime(2026, 3, 24) <= file_date <= datetime(2026, 5, 11):
                    should_load = True
            
            # 2. Orlando e Paris Studios: Todo Março, Abril e até 11/05
            elif park_key in ["universal_studios_orlando", "walt_disney_studios_paris"]:
                if datetime(2026, 3, 1) <= file_date <= datetime(2026, 5, 11):
                    should_load = True
            
            # 3. Demais arquivos (Se você quiser carregar outros que faltaram no crash)
            # Caso queira carregar TUDO que crashou em Abril para todos os parques:
            elif datetime(2026, 4, 1) <= file_date <= datetime(2026, 4, 30):
                 should_load = True

            if not should_load:
                continue

            # --- PROCESSAMENTO ---
            df = pd.read_csv(file_path)
            
            df = df.rename(columns={
                'id': 'ride_id',
                'name': 'ride_name',
                'collected_at': 'timestamp_utc'
            })
            
            df['park_id'] = config['id']
            df['park_name'] = config['name']
            df['timezone'] = config['tz']
            df['land'] = df['land'].fillna('N/A') if 'land' in df.columns else 'N/A'

            df_final = df[[
                "park_id", "park_name", "land", "ride_id", 
                "ride_name", "wait_time", "is_open", 
                "timestamp_utc", "timezone"
            ]]

            batch_list.append(df_final)
            processed_count += 1

            # Upload a cada 100 arquivos para economizar cota
            if len(batch_list) >= 100:
                upload_batch_to_bq(batch_list)
                batch_list = []

        except Exception as e:
            print(f"❌ Erro em {filename}: {e}")

    # Upload do restante
    if batch_list:
        upload_batch_to_bq(batch_list)

    print(f"\n🏁 Processo concluído! {processed_count} arquivos enviados para a tabela temporária.")

🔍 Analisando 3887 arquivos...
📦 Lote de 100 arquivos carregado (660778 linhas).
📦 Lote de 100 arquivos carregado (667141 linhas).
📦 Lote de 100 arquivos carregado (573010 linhas).
📦 Lote de 100 arquivos carregado (493572 linhas).
📦 Lote de 100 arquivos carregado (333268 linhas).
📦 Lote de 100 arquivos carregado (793348 linhas).
📦 Lote de 73 arquivos carregado (501323 linhas).

🏁 Processo concluído! 673 arquivos enviados para a tabela temporária.
